# Exercise 2: Statistical Inference

Statistical inference helps us answer questions about a population using a sample. Because estimates vary from sample to sample, we need to understand how much of an observed result could reflect sampling variation. Confidence intervals express uncertainty about an estimate, while hypothesis tests assess specific claims about the population.

We use two datasets for different purposes. First, the hourly bike-rental records in `data/bike-hours.csv` serve as a complete population. Its mean is known, so we can draw samples and check how well our methods recover it. We then apply these methods to the records of 649 students taking Portuguese at two secondary schools in Portugal, in `data/student-por.csv`. Here we investigate average grades, differences between schools, and relationships between student characteristics.

The analysis follows four steps:

- study how statistics vary across samples,
- build confidence intervals using formulas and the bootstrap,
- choose hypothesis tests to suit questions about means and categorical relationships, and
- examine the risk of false findings when testing many claims and apply multiple-comparison corrections.

## Setup

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA = Path("data")
rng = np.random.default_rng(2026)

In [ ]:
# The course style. The colours repeat template/themes/data-science.css, so a change there is repeated here.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.facecolor": GROUND,
    "figure.constrained_layout.use": True,
    "savefig.facecolor": GROUND,
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Archivo", "DejaVu Sans"],
    "font.size": 12,
    "text.color": INK,
    "axes.facecolor": GROUND,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "grid.color": GRID,
    "xtick.color": INK,
    "ytick.color": INK,
    "legend.frameon": False,
    "image.cmap": "cividis",
})
sns.set_palette(PALETTE)

## 1. Sampling

A population is the whole set of units under study, and a sample is the part that is observed. Here the population is every hour of two years of bike rentals, so its mean $\mu$ and its standard deviation $\sigma$ are known exactly.

In [ ]:
bikes = pd.read_csv(DATA / "bike-hours.csv")
population = bikes["rentals"].to_numpy()
mu, sigma = population.mean(), population.std()
len(population), round(mu, 1), round(sigma, 1)

In [ ]:
sample = rng.choice(population, size=30)
round(sample.mean(), 1), np.median(sample)

A second sample of 30 hours gives a different mean and a different median. Drawing many samples and computing the statistic on each builds its sampling distribution.

In [ ]:
DRAWS, N = 5000, 30
samples = rng.choice(population, size=(DRAWS, N))
means, medians = samples.mean(axis=1), np.median(samples, axis=1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(population, bins=40, color=INK)
axes[0].set(title="Population of hours", xlabel="Rentals per hour")
axes[1].hist(means, bins=40, color=PALETTE[0])
axes[1].set(title=f"Means of {N} hours", xlabel="Sample mean")
axes[2].hist(medians, bins=40, color=PALETTE[2])
axes[2].set(title=f"Medians of {N} hours", xlabel="Sample median")
for ax, value in zip(axes, [mu, mu, np.median(population)]):
    ax.axvline(value, color=PALETTE[1], linewidth=2)
plt.show()

The population is skewed, and the sample means gather around $\mu$ in a much narrower and more symmetric shape. The standard deviation of the sampling distribution is the standard error, and for the mean it equals $\sigma / \sqrt{n}$.

In [ ]:
round(means.std(), 2), round(sigma / np.sqrt(N), 2)

## 2. The Central Limit Theorem

For independent draws from a population with a finite variance, the distribution of the sample mean approaches a normal distribution with mean $\mu$ and standard deviation $\sigma / \sqrt{n}$ as $n$ grows. The panels draw the means of samples of four sizes against that normal curve.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, n in zip(axes, [2, 5, 30, 100]):
    sample_means = rng.choice(population, size=(DRAWS, n)).mean(axis=1)
    ax.hist(sample_means, bins=40, density=True, color=PALETTE[0])
    grid = np.linspace(sample_means.min(), sample_means.max(), 200)
    ax.plot(grid, stats.norm.pdf(grid, mu, sigma / np.sqrt(n)), color=PALETTE[1], linewidth=2)
    ax.set(title=f"n = {n}, skew {stats.skew(sample_means):.2f}", xlabel="Sample mean", yticks=[])
plt.show()

At $n = 2$ the means keep the skew of the population. By $n = 30$ they are close to the normal curve, and by $n = 100$ the skew is nearly gone. A more strongly skewed population needs a larger sample before the approximation holds.



## 3. Confidence Intervals

A sample mean is an estimate of the population mean, and another sample would give a different estimate. A confidence interval expresses this sampling uncertainty as an estimate plus or minus a margin of error. It estimates the population mean, rather than the spread of individual observations.

A 95 % confidence interval for a mean is $\bar{x} \pm t_{0.975,\,n-1} \, s / \sqrt{n}$, where $\bar{x}$ is the sample mean, $s / \sqrt{n}$ is its estimated standard error, and the $t$ quantile sets the margin for the chosen confidence level. The interval below uses the single sample of Section 1.

In [ ]:
stats.t.interval(0.95, df=N - 1, loc=sample.mean(), scale=stats.sem(sample))

One interval either contains $\mu$ or does not. The 95 % describes the procedure: over many samples, about 95 % of the intervals it builds contain $\mu$. Building one interval from each of the 5,000 samples measures how often that happens here.

In [ ]:
half_width = stats.t.ppf(0.975, N - 1) * samples.std(axis=1, ddof=1) / np.sqrt(N)
covered = np.abs(means - mu) <= half_width
covered.mean()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
shown = np.arange(100)
colours = np.where(covered[shown], PALETTE[0], PALETTE[1])
ax.vlines(shown, means[shown] - half_width[shown], means[shown] + half_width[shown], colors=colours, linewidth=2)
ax.axhline(mu, color=INK, linewidth=1.5)
ax.set(title=f"100 intervals from samples of {N} hours, {(~covered[shown]).sum()} miss the mean",
       xlabel="Sample", ylabel="Rentals per hour")
plt.show()

The coverage falls a little short of 95 %, because the population is skewed and 30 hours do not make the sample mean fully normal. The confidence level holds exactly only under the assumptions of the procedure.

## 4. The Students

Each row is one student, with the school, background, and habits, the number of absences, and the grades `G1` and `G2` of the two periods and the final grade `G3`. Grades run from 0 to 20, and 10 is the pass mark.

`GP` is Gabriel Pereira school, and `MS` is Mousinho da Silveira school. In `address`, `U` means urban and `R` means rural.

In [ ]:
students = pd.read_csv(DATA / "student-por.csv", sep=";")
columns = ["school", "sex", "address", "Mjob", "absences", "G1", "G2", "G3"]
students.shape, students[columns].head()

In [ ]:
students.loc[students["G3"] == 0, ["school", "G1", "G2", "G3", "absences"]]

Fifteen students have a final grade of 0, and every one of them has zero recorded absences, against about a third of the other students. Seven of them also have a second-period grade of 0. These rows describe students who left the course before the final rather than students who earned 0 on it. A test of the final grade asks about the students who sat the final, so the analysis removes these rows.

In [ ]:
grades = students[students["G3"] > 0].reset_index(drop=True)
len(grades)

For the student records, the population mean is unknown. The same interval formula as in Section 3 expresses uncertainty about the mean final grade estimated from these students.

In [ ]:
g3 = grades["G3"]
stats.t.interval(0.95, df=len(g3) - 1, loc=g3.mean(), scale=stats.sem(g3))

## 5. The Bootstrap

The bootstrap builds an interval for any statistic without a formula. It treats the sample as a stand-in for the population, draws resamples of the same size with replacement, computes the statistic on each, and reads the interval from the resulting distribution. The percentile interval takes the 2.5 % and 97.5 % quantiles of that distribution.

In [ ]:
absences = grades["absences"].to_numpy()
percentile = stats.bootstrap((absences,), np.mean, n_resamples=9999, method="percentile", rng=rng)
t_interval = stats.t.interval(0.95, df=len(absences) - 1, loc=absences.mean(), scale=stats.sem(absences))
pd.DataFrame({"low": [percentile.confidence_interval.low, t_interval[0]],
              "high": [percentile.confidence_interval.high, t_interval[1]]},
             index=["percentile", "t"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(percentile.bootstrap_distribution, bins=50, color=PALETTE[5])
for i, bound in enumerate(percentile.confidence_interval):
    ax.axvline(bound, color=PALETTE[0], linewidth=2, label=None if i else "percentile interval")
ax.axvline(absences.mean(), color=INK, linestyle="--", label="sample mean")
ax.set(title="Bootstrap distribution of the mean number of absences", xlabel="Mean absences", yticks=[])
ax.legend()
plt.show()

The absences have a long right tail, but the mean of 634 students is close to normal, so the percentile interval lies close to the t-interval. The bootstrap gains over the formula on a statistic that has no simple formula for its standard error. It only reflects the sample, though, and the median of a column with many repeated values barely moves between resamples.

In [ ]:
stats.bootstrap((absences,), np.median, n_resamples=9999, method="percentile", rng=rng).confidence_interval

## 6. Reading a p-Value

Two samples can have different means even when their population means are equal. A hypothesis test asks how unusual the observed result would be under a specific claim, such as equal population means. A small p-value provides evidence against that claim, under the test's assumptions.

Every test in this session follows the same five steps:

1. state the null hypothesis $H_0$ and the alternative $H_1$ before looking at the result,
2. choose the significance level $\alpha$, here 0.05,
3. compute a test statistic that measures how far the data lie from what $H_0$ predicts,
4. compute the p-value, the probability under $H_0$ of a statistic at least as extreme as the observed one, and
5. reject $H_0$ when the p-value is below $\alpha$, and otherwise report that the data do not contradict it.

A p-value is not the probability that $H_0$ is true, and a large p-value does not show that $H_0$ is true. Rejecting a true $H_0$ is a Type I error, which happens with probability $\alpha$. Keeping a false $H_0$ is a Type II error, and one minus its probability is the power of the test. A simulation shows both: two samples of 30 hours come from the same population, and then the second comes from a population shifted by 150 rentals.

In [ ]:
first = rng.choice(population, size=(DRAWS, N))
same = rng.choice(population, size=(DRAWS, N))
shifted = rng.choice(population, size=(DRAWS, N)) + 150
p_null = stats.ttest_ind(first, same, axis=1, equal_var=False).pvalue
p_shift = stats.ttest_ind(first, shifted, axis=1, equal_var=False).pvalue
{"Type I error rate": (p_null < 0.05).mean(), "Type II error rate": (p_shift >= 0.05).mean()}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, p, title in [(axes[0], p_null, "H0 true: same population"), (axes[1], p_shift, "H0 false: shifted by 150")]:
    ax.hist(p, bins=20, range=(0, 1), color=PALETTE[0])
    ax.axvline(0.05, color=PALETTE[1], linewidth=2)
    ax.set(title=title, xlabel="p-value")
axes[0].set_ylabel("Simulations")
plt.show()

When $H_0$ is true, the p-values spread evenly between 0 and 1, so 5 % of them fall below 0.05 by chance. When it is false, they pile up near 0, and the share that does not reach 0.05 is the Type II error rate.

## 7. The t-Tests

Does the mean final grade differ from the pass mark of 10? The one-sample t-test compares a mean with a fixed value. Here $H_0$ states that the mean final grade equals 10.

In [ ]:
stats.ttest_1samp(g3, popmean=10)

The two-sample t-test compares the means of two independent groups. Welch's version does not assume equal variances, which suits two schools of different sizes and spreads. Like every t-test, it assumes independent observations and approximately normal means, which samples of 422 and 212 students grant. Here $H_0$ states that the mean final grade is the same at both schools.

In [ ]:
gp, ms = g3[grades["school"] == "GP"], g3[grades["school"] == "MS"]
welch = stats.ttest_ind(gp, ms, equal_var=False)
welch, welch.confidence_interval()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(grades, x="G3", y="school", color=PALETTE[5], ax=ax)
ax.set(title="Final grade by school", xlabel="Final grade", ylabel="")
plt.show()

The test rejects $H_0$. Students of `GP` score about 1.3 grade points higher on average, and the interval of the difference runs from 0.81 to 1.70 points. The test assesses the equal-means claim; the estimated difference and its interval describe the size of the difference and its uncertainty.

## 8. The Chi-Squared Tests

Is a student's school associated with an urban or rural home address? The chi-squared test of independence compares the observed counts of a contingency table with the counts expected if the variables were independent, and it needs an expected count of at least 5 in every cell.

In [ ]:
table = pd.crosstab(grades["school"], grades["address"])
result = stats.chi2_contingency(table)
table, result.pvalue, result.expected_freq.round(1)

The schools draw their students from different areas: about four in five students of `GP` live in a town, while about half of those of `MS` live in the country. The test of sex against the plan to continue to higher education does not reject $H_0$.

In [ ]:
stats.chi2_contingency(pd.crosstab(grades["sex"], grades["higher"])).pvalue

The goodness-of-fit test compares observed counts with the counts a model predicts. Mendel's model of inheritance predicts that a cross of two traits gives four kinds of seed in the ratio 9:3:3:1.

In [ ]:
mendel = pd.read_csv(DATA / "mendel.csv")
cross = mendel[mendel["cross"] == "shape and colour"]
expected = cross["ratio"] / cross["ratio"].sum() * cross["count"].sum()
stats.chisquare(cross["count"], f_exp=expected)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(cross))
ax.bar(x - 0.2, cross["count"], width=0.4, label="observed")
ax.bar(x + 0.2, expected, width=0.4, label="expected under 9:3:3:1")
ax.set(xticks=x, xticklabels=cross["phenotype"], ylabel="Seeds", title="Mendel's two-trait cross")
ax.legend()
plt.show()

The observed counts lie close to the expected ones, so the data do not contradict the model.

## 9. Multiple Comparisons

Each test at $\alpha = 0.05$ has a 5 % chance of a false rejection, so ten tests run together make at least one false rejection far more likely. The column `Mjob` records the occupation of each student's mother in five categories, and Welch's test of every pair of occupations gives a family of ten tests.

In [ ]:
jobs = sorted(grades["Mjob"].unique())
pairs = list(combinations(jobs, 2))
p_values = [stats.ttest_ind(g3[grades["Mjob"] == a], g3[grades["Mjob"] == b], equal_var=False).pvalue for a, b in pairs]
family = pd.DataFrame({"p": p_values}, index=[f"{a} - {b}" for a, b in pairs]).sort_values("p")
1 - 0.95 ** len(family)

The Bonferroni correction tests each p-value at $\alpha / m$, which holds the family-wise error rate, the chance of any false rejection, at $\alpha$. The Benjamini-Hochberg correction compares the $k$-th smallest p-value with $k\alpha / m$, which holds the false discovery rate, the expected share of false rejections among the rejections, and keeps more power.

In [ ]:
family["uncorrected"] = family["p"] < 0.05
family["bonferroni"] = multipletests(family["p"], alpha=0.05, method="bonferroni")[0]
family["benjamini-hochberg"] = multipletests(family["p"], alpha=0.05, method="fdr_bh")[0]
family.round(4)

In [ ]:
m = len(family)
rank = np.arange(1, m + 1)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(rank, family["p"], color=INK, zorder=3, label="sorted p-values")
ax.axhline(0.05, color=PALETTE[4], label="0.05, uncorrected")
ax.axhline(0.05 / m, color=PALETTE[1], label="Bonferroni, 0.05 / m")
ax.plot(rank, rank * 0.05 / m, color=PALETTE[0], label="Benjamini-Hochberg, k 0.05 / m")
ax.set(yscale="log", xlabel="Rank k", ylabel="p-value", title="Ten pairwise comparisons")
ax.legend()
plt.show()

Bonferroni keeps four of the seven uncorrected rejections. Benjamini-Hochberg keeps all seven, because the seventh smallest p-value lies below its threshold $7\alpha / m = 0.035$, and every smaller p-value is rejected with it. The choice follows the cost of an error: Bonferroni suits a family in which any false claim is costly, and Benjamini-Hochberg suits a screen whose discoveries are checked afterwards.